# LoRA Experiments — Qwen2-VL-2B on VQA-RADThis notebook runs all four LoRA experiments (1 pilot + 3 rank ablations) by toggling a single `EXPERIMENT` variable. All experiments share an identical training pipeline and evaluation protocol; only the LoRA rank, target modules, and training subset size differ.| EXPERIMENT | Train size | Epochs | Rank | Target modules | Trainable params | Use ||---|---:|---:|---:|---|---:|---|| `quick`  |  200 | 1 | 8 | attention only      | 2.18M | Pilot study (Member 1) || `rank4`  | 1797 | 3 | 4 | attention + FFN     | 4.62M | Ablation (best) || `rank8`  | 1797 | 3 | 8 | attention + FFN     | 9.23M | Ablation || `rank16` | 1797 | 3 | 16| attention + FFN     | 18.46M| Ablation |Set the `EXPERIMENT` variable in section 2 to choose which one to run.**Reproducibility**: all configs fix `seed=42`. Per-experiment results are saved to `checkpoints/lora_{EXPERIMENT}/`.**Hardware**: Colab T4 GPU. `quick` takes ~6 min, full ablation runs take ~3 hours each.

## 1. Environment setup

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Install pinned dependencies (~2-3 min)
%pip install -q --upgrade \
    torch==2.5.1 transformers==4.49.0 \
    peft==0.14.0 accelerate==1.2.1 bitsandbytes==0.45.0 \
    datasets==3.1.0 evaluate==0.4.3 sacrebleu==2.4.3 rouge-score==0.1.2 \
    pyyaml pillow scipy tqdm

In [ ]:
# Make project importable
import os, sys
PROJECT_ROOT = '/content/peft-medical-vqa'
os.chdir(PROJECT_ROOT)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
print(f"CWD: {os.getcwd()}")

## 2. Choose experimentSet `EXPERIMENT` to one of: `'quick'`, `'rank4'`, `'rank8'`, `'rank16'`.The notebook will load the matching config from `configs/` and write results to `checkpoints/lora_{EXPERIMENT}/`.

In [ ]:
# ┌─────────────────────────────────────────────┐
# │   Set this variable to choose experiment    │
# └─────────────────────────────────────────────┘
EXPERIMENT = 'rank4'   # one of: 'quick' / 'rank4' / 'rank8' / 'rank16'

CONFIGS = {
    'quick':  'configs/lora_quick.yaml',
    'rank4':  'configs/lora_rank4.yaml',
    'rank8':  'configs/lora_rank8.yaml',
    'rank16': 'configs/lora_rank16.yaml',
}

assert EXPERIMENT in CONFIGS, f"EXPERIMENT must be one of {list(CONFIGS.keys())}"
CONFIG_PATH = CONFIGS[EXPERIMENT]
print(f"Selected experiment: {EXPERIMENT}")
print(f"Config path: {CONFIG_PATH}")

## 3. Train

In [ ]:
from src.utils.seed import set_global_seed
from src.training.train_lora import LoRATrainingConfig, train_lora

cfg = LoRATrainingConfig.from_yaml(CONFIG_PATH)
set_global_seed(cfg.seed)
print(f"Train size: {cfg.max_train_samples or 'all 1797'}")
print(f"Epochs: {cfg.num_train_epochs}")
print(f"LoRA rank: {cfg.lora_r}")
print(f"Target modules: {cfg.target_modules}")
print(f"Output: {cfg.output_dir}")

In [ ]:
# Run training. Quick: ~6 min. Full ablation: ~3 hours on T4.
metrics = train_lora(cfg)

## 4. Inspect training metrics + loss curve

In [ ]:
import json
with open(f'{cfg.output_dir}/training_metrics.json') as f:
    m = json.load(f)

print(f"=== {EXPERIMENT} ===")
print(f"Trainable params: {m['params']['trainable']:,} ({m['params']['trainable_pct']:.4f}%)")
print(f"Train time: {m['training']['total_seconds']:.1f}s ({m['training']['total_seconds']/60:.1f} min)")
print(f"Peak GPU: {m['training']['peak_gpu_gb']:.2f} GB")
print(f"Final train loss: {m['training']['final_loss']:.4f}")
e = m['evaluation']
print(f"\nEval n={e['n']}")
print(f"Closed EM:   {e['closed']['exact_match']:.4f}  [{e['closed']['ci95']['lower']:.4f}, {e['closed']['ci95']['upper']:.4f}]")
print(f"Open F1:     {e['open']['f1']:.4f}  [{e['open']['ci95_f1']['lower']:.4f}, {e['open']['ci95_f1']['upper']:.4f}]")
print(f"Overall EM:  {e['overall']['exact_match']:.4f}  [{e['overall']['ci95']['lower']:.4f}, {e['overall']['ci95']['upper']:.4f}]")

In [ ]:
# Plot the loss curve
import matplotlib.pyplot as plt

losses = m['training']['loss_curve']
plt.figure(figsize=(8, 4))
plt.plot(losses)
plt.xlabel('Logging step')
plt.ylabel('Train loss')
plt.title(f'Training loss — {EXPERIMENT}')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Paired statistical tests vs zero-shot baselineCompare this LoRA run against the zero-shot baseline on the **same 451 test examples** using:- 10,000-resample paired bootstrap CIs on per-example score differences- McNemar's exact test on closed-form Overall EMBoth procedures account for shared variance (same examples scored by both models).

In [ ]:
from src.evaluation.statistical_tests import paired_bootstrap_ci_diff, mcnemar_test
import json, os

BASE_DIR = 'results/baseline'
LORA_DIR = cfg.output_dir
OUT_PATH = f'results/statistical_tests/baseline_vs_lora_{EXPERIMENT}.json'

with open(f'{BASE_DIR}/per_example_scores.json') as f:
    base = json.load(f)
with open(f'{LORA_DIR}/per_example_scores.json') as f:
    lora = json.load(f)

assert base['ids'] == lora['ids'], "Per-example IDs must match for paired tests"

results = {}
for metric_name, key in [
    ('Closed EM',   'correct'),
    ('Open ROUGE-L', 'rougeL'),
    ('Open BLEU-1',  'bleu1'),
    ('Overall EM',   'correct'),
]:
    if 'Closed' in metric_name:
        idx = [i for i, q in enumerate(base['qtypes']) if q == 'closed']
    elif 'Open' in metric_name:
        idx = [i for i, q in enumerate(base['qtypes']) if q == 'open']
    else:
        idx = list(range(len(base['ids'])))
    
    a = [base[key][i] for i in idx]
    b = [lora[key][i] for i in idx]
    
    boot = paired_bootstrap_ci_diff(a, b, n_resamples=10000, seed=42)
    results[metric_name] = {'paired_bootstrap': boot, 'n': len(idx)}
    
    if metric_name == 'Overall EM':
        results[metric_name]['mcnemar'] = mcnemar_test(a, b)

os.makedirs('results/statistical_tests', exist_ok=True)
with open(OUT_PATH, 'w') as f:
    json.dump({'experiment': EXPERIMENT, 'tests': results}, f, indent=2)

for metric, r in results.items():
    pb = r['paired_bootstrap']
    print(f"{metric:15s}  Δ = {pb['point']:+.4f}  [{pb['lower']:+.4f}, {pb['upper']:+.4f}]")
    if 'mcnemar' in r:
        print(f"  McNemar: b={r['mcnemar']['b']}, c={r['mcnemar']['c']}, p={r['mcnemar']['p_value']:.6f}")
print(f"\n✓ Saved to {OUT_PATH}")

## 6. What's nextAfter running each experiment (`quick`, `rank4`, `rank8`, `rank16`), the aggregated results are summarized in:- `results/tables/main_results.md` — Table 1 (main results) and Table 2 (statistical significance)- `results/figures/loss_curves.{png,pdf}` — training dynamics across all 3 ranks- `results/figures/rank_scaling.{png,pdf}` — rank vs accuracy (overfitting evidence)- `results/figures/improvements_bar.{png,pdf}` — per-rank improvements over baseline- `results/error_analysis/baseline_vs_lora_rank4.json` — qualitative wins (closed + open)For Members 2 (QLoRA) and 3 (DoRA), see `docs/HANDOFF.md` — both extensions require only 1-2 line changes to a config file.